In [ ]:
import torch
print(torch.cuda.is_available())
import pandas as pd, numpy as np
from sentence_transformers import SentenceTransformer


In [ ]:
from unsloth import FastModel
import torch

In [ ]:
model_name = "unsloth/gemma-3-12b-it-unsloth-bnb-4bit"

model, tokenizer = FastModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

In [5]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [6]:
from datasets import load_dataset, load_from_disk
from unsloth.chat_templates import standardize_data_formats

dataset = load_dataset("csv", data_files="progressive_train_80.csv")["train"]
dataset = standardize_data_formats(dataset)


In [ ]:
lexicon_df = pd.read_csv("./new_arabic_lexicon_17_07.csv")

lexicon_embeddings = np.load("./Generative/embeddings.npy")
embedding_model = SentenceTransformer('google/embeddinggemma-300m',
                               token = "your token")
for param in embedding_model.parameters():
    param.requires_grad = False

# 3. Switch to Evaluation Mode (Disables Dropout & Updates Batch Norm)
embedding_model.eval()

needed_categories = ["Past suicidal history", "Family suicide history", "Suicidal ideation", "Hopelessness", "Deliberate self harm", "Perceived burdensomeness"]


def build_query(msg):
    """
    given msg (str) build a query to the embedding gemma model (which works in query-document style)
    """
    prompt = "اي من هذه الجمل هي الاقرب من ناحية المضمون الى"
    prompt += f" '{msg}'"
    return prompt
    
def get_top_n_similar(msg, n=10):
    """
    given msg (str) get top n most similar phrases from the lexicon, then return their indices in the lexicon csv file
    to extract their categories later.
    output: list of integers (index of each phrase of the top n in the lexicon file)
    """
    global lexicon_embeddings
    query = build_query(msg)
    queries_embeddings = embedding_model.encode_query(query)
    similarities = pd.Series(embedding_model.similarity(queries_embeddings, lexicon_embeddings).flatten())
    return similarities.nlargest(n).index.tolist()


def categories_injection(user_input):
    """
    given user_input (str) get top 5 most similar phrases in the lexicon then extract their categories, and insert these categories
    into <context> tag to add it to the prompt later

    output: str containting <context> tag and catefories inside it. Example:  
        <context>
        Categories: Family suicide history, Preparatory acts, Family suicide history, Past suicidal history, Family suicide history
        </context>
    """
    top_5_categories = get_top_n_similar(user_input, 5)
    top_5_categories = lexicon_df.iloc[top_5_categories]['Category']
    
    formatted_cats = ", ".join(top_5_categories)
    categories_context = f"<context>\nCategories: {formatted_cats}\n</context>\n\n"
    return categories_context


def apply_chat_template(example):
    categories_as_context = categories_injection(str(example["input"]))

    return {
        "text": tokenizer.apply_chat_template(
            [{"role": "user", "content": categories_as_context + str(example["input"])},
             {"role": "assistant", "content": str(example["output"])}],
            tokenize=False
        )
    }

dataset = dataset.map(apply_chat_template)


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,  # No accumulation, update after every example
        warmup_steps = 0,
        num_train_epochs = 1,
        max_steps = 2000,  # Or remove it to let epochs control training
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        save_steps=500,
        save_strategy = "steps"
    )
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

In [14]:
torch.cuda.empty_cache()  # Clear GPU memory

In [ ]:
trainer_stats = trainer.train()

In [14]:
new_model_local = "Gemma-3-12B-it-2000steps_Arapro_with_lexicon"
model.save_pretrained(new_model_local) # Local saving
tokenizer.save_pretrained(new_model_local)

['Gemma-3-12B-it-FirstResponder_progressive_train_80_500steps/processor_config.json']